# 1. Inventory

## 1.a. Workspaces

In [ ]:
import sempy.fabric as fabric
import pandas as pd

client = fabric.FabricRestClient()

# --------------------------------------------------
# Workspace Runtime Inventory
# --------------------------------------------------

workspace_results = []

workspaces = fabric.list_workspaces()

for _, ws in workspaces.iterrows():

    workspace_id = ws["Id"]
    workspace_name = ws["Name"]

    try:

        r = client.get(
            f"/v1/workspaces/{workspace_id}/spark/settings"
        )

        if r.status_code != 200:
            continue

        runtime = (
            r.json()
             .get("environment", {})
             .get("runtimeVersion")
        )

        workspace_results.append({
            "WorkspaceName": workspace_name,
            "WorkspaceId": workspace_id,
            "RuntimeVersion": runtime
        })

    except Exception as ex:

        workspace_results.append({
            "WorkspaceName": workspace_name,
            "WorkspaceId": workspace_id,
            "RuntimeVersion": f"ERROR: {str(ex)}"
        })

workspace_df = pd.DataFrame(workspace_results)

runtime12_workspaces = workspace_df[
    workspace_df["RuntimeVersion"] == "1.2"
]

print(f"Workspaces Runtime 1.2 : {len(runtime12_workspaces)}")

display(runtime12_workspaces)

## 1.b. Environments

In [ ]:
environment_results = []

for _, ws in workspaces.iterrows():

    workspace_id = ws["Id"]
    workspace_name = ws["Name"]

    try:

        envs = client.get(
            f"/v1/workspaces/{workspace_id}/environments"
        )

        if envs.status_code != 200:
            continue

        for env in envs.json().get("value", []):

            env_id = env["id"]
            env_name = env["displayName"]

            try:

                compute = client.get(
                    f"/v1/workspaces/{workspace_id}/environments/{env_id}/staging/sparkcompute?beta=false"
                )

                if compute.status_code != 200:
                    continue

                runtime = compute.json().get(
                    "runtimeVersion"
                )

                environment_results.append({
                    "WorkspaceName": workspace_name,
                    "WorkspaceId": workspace_id,
                    "EnvironmentName": env_name,
                    "EnvironmentId": env_id,
                    "RuntimeVersion": runtime
                })

            except Exception as ex:

                environment_results.append({
                    "WorkspaceName": workspace_name,
                    "WorkspaceId": workspace_id,
                    "EnvironmentName": env_name,
                    "EnvironmentId": env_id,
                    "RuntimeVersion": f"ERROR: {str(ex)}"
                })

    except Exception:
        pass

environment_df = pd.DataFrame(environment_results)

runtime12_environments = environment_df[
    environment_df["RuntimeVersion"] == "1.2"
]

print(
    f"Environments Runtime 1.2 : {len(runtime12_environments)}"
)

display(runtime12_environments)

# 2. Migration

In [ ]:
# ============================================================
# Fabric Runtime Migration Notebook
#
# Objectif :
# - Mettre à jour les Workspaces vers TARGET_RUNTIME
# - Mettre à jour les Environment Items
# - Publier les Environment Items
# - Capturer toutes les erreurs dans un rapport
# - Exporter les résultats dans le Lakehouse
#
# Documentation :
# https://learn.microsoft.com/en-us/fabric/data-engineering/runtime
# https://learn.microsoft.com/en-us/fabric/data-engineering/environment-public-api
# ============================================================

import sempy.fabric as fabric
import pandas as pd
import time

# ============================================================
# PARAMETRES
# ============================================================

TARGET_RUNTIME = "2.0"

# Optionnel : limiter à certains workspaces
TARGET_WORKSPACES = [
    # "Fabric-DEV",
    # "Fabric-UAT",
    # "Fabric-PROD"
]

# ============================================================
# INITIALISATION
# ============================================================

client = fabric.FabricRestClient()

migration_errors = []
migration_success = []

# ============================================================
# WORKSPACE UPGRADE
# ============================================================

def upgrade_workspace_runtime(
    workspace_name,
    workspace_id
):

    try:

        payload = {
            "environment": {
                "runtimeVersion": TARGET_RUNTIME
            }
        }

        r = client.patch(
            f"/v1/workspaces/{workspace_id}/spark/settings",
            json=payload
        )

        if r.status_code in [200, 202]:

            migration_success.append({
                "WorkspaceName": workspace_name,
                "EnvironmentName": None,
                "EnvironmentId": None,
                "Operation": "WorkspaceRuntimeUpdate"
            })

            print(
                f"Workspace updated : {workspace_name}"
            )

            return True

        migration_errors.append({
            "WorkspaceName": workspace_name,
            "EnvironmentName": None,
            "EnvironmentId": None,
            "Operation": "WorkspaceRuntimeUpdate",
            "StatusCode": r.status_code,
            "Error": r.text
        })

        return False

    except Exception as ex:

        migration_errors.append({
            "WorkspaceName": workspace_name,
            "EnvironmentName": None,
            "EnvironmentId": None,
            "Operation": "WorkspaceRuntimeUpdate",
            "StatusCode": "Exception",
            "Error": str(ex)
        })

        return False


# ============================================================
# LIST ENVIRONMENTS
# ============================================================

def get_environments(workspace_id):

    try:

        r = client.get(
            f"/v1/workspaces/{workspace_id}/environments"
        )

        if r.status_code != 200:
            return []

        return r.json().get("value", [])

    except Exception:
        return []


# ============================================================
# GET ENVIRONMENT COMPUTE CONFIG
# ============================================================

def get_environment_compute(
    workspace_id,
    environment_id
):

    try:

        r = client.get(
            f"/v1/workspaces/{workspace_id}/environments/{environment_id}/staging/sparkcompute?beta=false"
        )

        if r.status_code != 200:
            return None

        return r.json()

    except Exception:
        return None


# ============================================================
# UPDATE ENVIRONMENT RUNTIME
# ============================================================

def upgrade_environment_runtime(
    workspace_name,
    workspace_id,
    environment_name,
    environment_id,
    compute
):

    try:

        # Copie de sécurité
        payload = compute.copy()

        # Modification du runtime uniquement
        payload["runtimeVersion"] = TARGET_RUNTIME

        r = client.patch(
            f"/v1/workspaces/{workspace_id}/environments/{environment_id}/staging/sparkcompute?beta=false",
            json=payload
        )

        if r.status_code == 200:

            migration_success.append({
                "WorkspaceName": workspace_name,
                "EnvironmentName": environment_name,
                "EnvironmentId": environment_id,
                "Operation": "EnvironmentRuntimeUpdate"
            })

            print(
                f"Environment updated : {environment_name}"
            )

            return True

        migration_errors.append({
            "WorkspaceName": workspace_name,
            "EnvironmentName": environment_name,
            "EnvironmentId": environment_id,
            "Operation": "EnvironmentRuntimeUpdate",
            "StatusCode": r.status_code,
            "Error": r.text
        })

        return False

    except Exception as ex:

        migration_errors.append({
            "WorkspaceName": workspace_name,
            "EnvironmentName": environment_name,
            "EnvironmentId": environment_id,
            "Operation": "EnvironmentRuntimeUpdate",
            "StatusCode": "Exception",
            "Error": str(ex)
        })

        return False


# ============================================================
# PUBLISH ENVIRONMENT
# ============================================================

def publish_environment(
    workspace_name,
    workspace_id,
    environment_name,
    environment_id
):

    try:

        r = client.post(
            f"/v1/workspaces/{workspace_id}/environments/{environment_id}/staging/publish?beta=false"
        )

        if r.status_code in [200, 202]:

            migration_success.append({
                "WorkspaceName": workspace_name,
                "EnvironmentName": environment_name,
                "EnvironmentId": environment_id,
                "Operation": "PublishEnvironment"
            })

            print(
                f"Publish started : {environment_name}"
            )

            return True

        migration_errors.append({
            "WorkspaceName": workspace_name,
            "EnvironmentName": environment_name,
            "EnvironmentId": environment_id,
            "Operation": "PublishEnvironment",
            "StatusCode": r.status_code,
            "Error": r.text
        })

        return False

    except Exception as ex:

        migration_errors.append({
            "WorkspaceName": workspace_name,
            "EnvironmentName": environment_name,
            "EnvironmentId": environment_id,
            "Operation": "PublishEnvironment",
            "StatusCode": "Exception",
            "Error": str(ex)
        })

        return False


# ============================================================
# WORKSPACE SELECTION
# ============================================================

all_workspaces = fabric.list_workspaces()

if len(TARGET_WORKSPACES) > 0:

    workspaces = all_workspaces[
        all_workspaces["Name"].isin(
            TARGET_WORKSPACES
        )
    ]

else:

    workspaces = all_workspaces

print(
    f"Selected workspaces : {len(workspaces)}"
)

# ============================================================
# MAIN PROCESS
# ============================================================

for _, ws in workspaces.iterrows():

    workspace_id = ws["Id"]
    workspace_name = ws["Name"]

    print(
        f"\n{'='*70}"
    )

    print(
        f"Processing workspace : {workspace_name}"
    )

    print(
        f"{'='*70}"
    )

    # --------------------------------------------------------
    # Workspace Runtime
    # --------------------------------------------------------

    upgrade_workspace_runtime(
        workspace_name,
        workspace_id
    )

    # --------------------------------------------------------
    # Environments
    # --------------------------------------------------------

    environments = get_environments(
        workspace_id
    )

    print(
        f"Environments found : {len(environments)}"
    )

    for env in environments:

        env_id = env["id"]
        env_name = env["displayName"]

        print(
            f"\nEnvironment : {env_name}"
        )

        compute = get_environment_compute(
            workspace_id,
            env_id
        )

        if compute is None:

            migration_errors.append({
                "WorkspaceName": workspace_name,
                "EnvironmentName": env_name,
                "EnvironmentId": env_id,
                "Operation": "ReadCompute",
                "StatusCode": "Failed",
                "Error": "Unable to read compute configuration"
            })

            continue

        current_runtime = compute.get(
            "runtimeVersion",
            "unknown"
        )

        print(
            f"Current runtime : {current_runtime}"
        )

        if current_runtime == TARGET_RUNTIME:

            print(
                "Already on target runtime"
            )

            continue

        # ----------------------------------------------------
        # Update Runtime
        # ----------------------------------------------------

        updated = upgrade_environment_runtime(
            workspace_name,
            workspace_id,
            env_name,
            env_id,
            compute
        )

        if not updated:
            continue

        # ----------------------------------------------------
        # Publish
        # ----------------------------------------------------

        publish_environment(
            workspace_name,
            workspace_id,
            env_name,
            env_id
        )

        time.sleep(2)

# ============================================================
# REPORTS
# ============================================================

error_df = pd.DataFrame(
    migration_errors
)

success_df = pd.DataFrame(
    migration_success
)

print("\n")
print("=" * 70)
print("MIGRATION SUMMARY")
print("=" * 70)

print(
    f"Success count : {len(success_df)}"
)

print(
    f"Error count : {len(error_df)}"
)

# ============================================================
# DISPLAY RESULTS
# ============================================================

if len(error_df) > 0:

    print("\nERRORS")

    display(error_df)

if len(success_df) > 0:

    print("\nSUCCESS")

    display(success_df)